In [ ]:
import os
os.chdir('mmsegmentation')

In [ ]:
import time, statistics, numpy as np, torch
from mmengine.config import Config
from mmseg.utils import register_all_modules
from mmseg.registry import MODELS, DATASETS
from mmengine.runner import load_checkpoint
from mmengine.dataset import default_collate

# === RUTAS ===
cfg_path = 'configs/Resnet/Resnet_test.py'
checkpoint_path = '/work_dirs/checkpoints.pth'


# === SETUP RENDIMIENTO A100 ===
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# === CARGA CONFIG / REGISTRO / MODELO ===
cfg = Config.fromfile(cfg_path)
register_all_modules()
model = MODELS.build(cfg.model).cuda().eval()
load_checkpoint(model, checkpoint_path, map_location='cuda')

# === DATASET VALIDACIÓN ===
dl_cfg = cfg.get('val_dataloader', None) or cfg.get('test_dataloader', None)
assert dl_cfg is not None, "Tu config no tiene val_dataloader ni test_dataloader."
val_dataset_cfg = dl_cfg['dataset']
val_dataset = DATASETS.build(val_dataset_cfg)
N_DS = len(val_dataset)


# === PARÁMETROS DE MEDICIÓN ===
N_WARM = min(30, N_DS)            # warm-up sobre parte del val
N_REPEATS = 10                    # repite todo el val N veces
TIMING_MODE = 'model'             # 'model' (solo modelo) o 'e2e' (end-to-end)
PRECISION = 'fp16'                # 'fp32' | 'fp16' | 'bf16'

def autocast_ctx():
    if PRECISION == 'fp32':  return torch.autocast('cuda', dtype=torch.float32)
    if PRECISION == 'fp16':  return torch.autocast('cuda', dtype=torch.float16)
    if PRECISION == 'bf16':  return torch.autocast('cuda', dtype=torch.bfloat16)
    raise ValueError('PRECISION inválida')

def to_cuda_batch(data):
    batch = default_collate([data])
    batch = {k: v.cuda() if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
    batch['data_samples'] = [ds.to('cuda') for ds in batch['data_samples']]
    return batch

# === WARM-UP ===
with torch.inference_mode(), autocast_ctx():
    for i in range(N_WARM):
        data = val_dataset[i]
        batch = to_cuda_batch(data)
        _ = model.test_step(batch)

# === MEDICIÓN ===
times_ms = []
total_imgs = N_DS * N_REPEATS
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

with torch.inference_mode(), autocast_ctx():
    for r in range(N_REPEATS):
        for i in range(N_DS):
            data = val_dataset[i]

            if TIMING_MODE == 'model':
                # Prepara fuera del cronómetro → mide solo forward/post del modelo
                batch = to_cuda_batch(data)
                torch.cuda.synchronize()
                t0 = time.perf_counter()
                _ = model.test_step(batch)
                torch.cuda.synchronize()
                t1 = time.perf_counter()

            elif TIMING_MODE == 'e2e':
                # Incluye collate + .cuda() dentro del cronómetro
                torch.cuda.synchronize()
                t0 = time.perf_counter()
                batch = to_cuda_batch(data)
                _ = model.test_step(batch)
                torch.cuda.synchronize()
                t1 = time.perf_counter()

            else:
                raise ValueError('TIMING_MODE debe ser "model" o "e2e"')

            times_ms.append((t1 - t0) * 1000.0)

# === RESULTADOS ===
avg = sum(times_ms) / len(times_ms)
p50 = statistics.median(times_ms)
p95 = float(np.percentile(times_ms, 95))
fps_media = 1000.0 / avg
fps_p50   = 1000.0 / p50
fps_p95   = 1000.0 / p95

mem_alloc = torch.cuda.max_memory_allocated() / (1024**2)
mem_res   = torch.cuda.max_memory_reserved() / (1024**2)

print(f"\n🚀 INFERENCIA COMPLETADA: {total_imgs} pasadas ({N_DS} imgs × {N_REPEATS} rep)")
print(f"Modo: {TIMING_MODE} | Precisión: {PRECISION}")
print(f"Lat media: {avg:.2f} ms | p50: {p50:.2f} ms | p95: {p95:.2f} ms")
print(f"FPS promedio: {fps_media:.1f} | FPS@p50≈ {fps_p50:.1f} | FPS@p95≈ {fps_p95:.1f}")
print(f"Memoria pico (alloc): {mem_alloc:.1f} MiB | Pico reservado: {mem_res:.1f} MiB")